# Create dataset using preprocessed data

In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [2]:
from datetime import datetime

from tqdm import tqdm

# convert dates to datetime objects, and PTID and Voltage to integers
date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

100%|██████████| 74063/74063 [00:00<00:00, 114582.65it/s]


{'PTID': 26053,
 'Name': 'MOUNTAIN-SWANROAD_115_104-3',
 'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
 'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
 'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
 'Voltage': 115,
 'FirstBus': 'MOUNTAIN',
 'SecondBus': 'SWANROAD',
 'OutageType': 'Planned'}

In [3]:
from pathlib import Path

# create output directory if it doesn't exist
output_path = Path("output")
output_path.mkdir(exist_ok=True)

# HYPER PARAMETERS

In [4]:
EVENT_WINDOW_HOURS_VALUES = [0.25, 0.5, *range(1, 13)]
MIN_YEAR = 2008

VOLTAGES = [69, 115, 132, 220, 345, 500, 735]
VOLTAGE_GROUP = {
    # 69 or less
    27: 69,
    34: 69,
    69: 69,
    # 155/120
    115: 115,
    120: 115,
    # 132/138
    132: 132,
    138: 132,
    # 220/230
    220: 220,
    230: 220,
    # 345
    345: 345,
    # 500
    500: 500,
    # 735/765
    735: 735,
    765: 735,
}

INTERVALS_MINUTES = [15, 30, 60, 120]

In [5]:
# remove old data and sort by MinTimeStamp
actual_data = sorted(
    [
        row
        for row in actual_data
        if row["MinTimeStamp"].year >= MIN_YEAR and row["OutDatetime"].year >= MIN_YEAR
    ],
    key=lambda x: x["MinTimeStamp"],
)

In [6]:
# load graph
import igraph

g = igraph.Graph.Read_Pickle("res/outage_graph.pkl")

In [7]:
# load buses/nodes information
import json

bus_name_to_index_path = Path("res/bus_name_to_index.json")
bus_name_to_index = json.loads(bus_name_to_index_path.read_text())

In [8]:
import random
from collections import deque

import igraph as ig
import numpy as np
import pandas as pd


def assign_line_zones(
    g: ig.Graph,
    seed: int = 42,
    objective_function: str = "modularity",
    inter_zone_policy: str = "boundary_pair",
):
    """
    Assign line zones using community structure detected by Leiden.

    Parameters
    ----------
    g : igraph.Graph
        Graph whose vertices represent buses and whose edges represent lines.
    seed : int, default 42
        Random seed for deterministic community detection.
    objective_function : str, default "modularity"
        Objective function passed to `community_leiden`.
    inter_zone_policy : str, default "boundary_pair"
        How to label edges that connect vertices in different communities.
        One of "boundary", "boundary_pair", or "lower_zone".

    Returns
    -------
    communities : igraph.clustering.VertexClustering
        Vertex clustering produced by Leiden.
    edge_zone_df : pandas.DataFrame
        DataFrame with line zone assignments and inter-zone flags.
    """
    if inter_zone_policy not in {"boundary", "boundary_pair", "lower_zone"}:
        raise ValueError(
            "inter_zone_policy must be one of: "
            "'boundary', 'boundary_pair', or 'lower_zone'"
        )

    random.seed(seed)
    np.random.seed(seed)

    # run community detection using Leiden algorithm
    communities = g.community_leiden(objective_function=objective_function)
    raw_membership = list(communities.membership)
    community_sizes = pd.Series(raw_membership).value_counts()
    # mapping from node id to zone id
    old_to_new = {
        old_cid: new_cid
        for new_cid, old_cid in enumerate(community_sizes.index.tolist())
    }
    # use python capabilities to assign zone id to noes in the same igraph object
    vertex_zone = [old_to_new[cid] for cid in raw_membership]
    g.vs["zone_id"] = vertex_zone

    edge_records = []
    for eid, edge in enumerate(g.es):
        u, v = edge.tuple
        zone_u = vertex_zone[u]
        zone_v = vertex_zone[v]
        is_inter_zone = zone_u != zone_v

        if not is_inter_zone:
            line_zone = zone_u
        elif inter_zone_policy == "boundary":
            line_zone = -1
        elif inter_zone_policy == "boundary_pair":
            line_zone = f"{min(zone_u, zone_v)}--{max(zone_u, zone_v)}"
        else:
            line_zone = min(zone_u, zone_v)

        edge["zone_id"] = line_zone
        edge["is_inter_zone"] = is_inter_zone
        edge_records.append(
            {
                "edge_id": eid,
                "from_vertex": u,
                "to_vertex": v,
                "from_name": g.vs[u]["name"] if "name" in g.vs.attributes() else u,
                "to_name": g.vs[v]["name"] if "name" in g.vs.attributes() else v,
                "from_zone": zone_u,
                "to_zone": zone_v,
                "line_zone": line_zone,
                "is_inter_zone": is_inter_zone,
            }
        )

    return communities, pd.DataFrame(edge_records)


def build_zone_distance_lookup(edge_zone_df):
    """
    Build a zone-to-zone distance lookup from line zone assignments.

    Parameters
    ----------
    edge_zone_df : pandas.DataFrame
        DataFrame containing line zone assignments with `from_zone` and
        `to_zone` columns.

    Returns
    -------
    dict[int, dict[int, int]]
        Mapping from each source zone to a mapping of target zones and
        their minimum number of hops in the zone adjacency graph.
    """
    zones = sorted(
        set(edge_zone_df["from_zone"].astype(int))
        | set(edge_zone_df["to_zone"].astype(int))
    )
    adjacency = {zone: {zone} for zone in zones}

    for from_zone, to_zone in zip(
        edge_zone_df["from_zone"].astype(int),
        edge_zone_df["to_zone"].astype(int),
    ):
        adjacency.setdefault(from_zone, {from_zone}).add(to_zone)
        adjacency.setdefault(to_zone, {to_zone}).add(from_zone)

    distances = {}
    for source in adjacency:
        source_distances = {source: 0}
        queue = deque([source])
        while queue:
            current = queue.popleft()
            for neighbor in adjacency[current]:
                if neighbor not in source_distances:
                    source_distances[neighbor] = source_distances[current] + 1
                    queue.append(neighbor)
        distances[source] = source_distances

    return distances


communities, edge_zone_df = assign_line_zones(
    g,
    seed=42,
    objective_function="modularity",
    inter_zone_policy="boundary_pair",
)
edge_zone_df.to_csv(output_path / "edge_zones.csv", index=False)

ZONE_IDS = sorted(set(g.vs["zone_id"]))
NUM_ZONES = len(ZONE_IDS)
ZONE_ID_TO_POSITION = {zone_id: idx for idx, zone_id in enumerate(ZONE_IDS)}
BUS_ZONE = {
    bus_name: int(g.vs[bus_index]["zone_id"])
    for bus_name, bus_index in bus_name_to_index.items()
}
ZONE_DISTANCE_LOOKUP = build_zone_distance_lookup(edge_zone_df)
MAX_ZONE_DISTANCE = max(
    distance
    for source_distances in ZONE_DISTANCE_LOOKUP.values()
    for distance in source_distances.values()
)


def row_zones(row):
    return BUS_ZONE[row["FirstBus"]], BUS_ZONE[row["SecondBus"]]

In [9]:
from datetime import timedelta

import numpy as np


def create_when_sample(ref_row, window, last_scheduled):
    """Create one supervised sample for the when-prediction dataset.

    Parameters
    ----------
    ref_row : mapping-like
        Reference outage row whose outage type, time-to-event, and zones become labels.
    window : sequence of mapping-like
        Prior outage rows in the lookback window used to build event-count, voltage,
        timing, and graph-based features.
    last_scheduled : mapping-like
        Most recent scheduled outage row used to build planned-outage context features.

    Returns
    -------
    tuple[list, list]
        Feature vector and label vector for a single sample.
    """
    ############
    # features #
    ############
    # simple counting features
    num_events = len(window)
    num_unique_ptids = len(set(row["PTID"] for row in window))
    voltage_group_list = {i: 0 for i in VOLTAGES}
    for row in window:
        voltage_group_list[VOLTAGE_GROUP[row["Voltage"]]] += 1
    voltage_group_list = [voltage_group_list[v] for v in VOLTAGES]
    num_planned = len([row for row in window if row["OutageType"] == "Planned"])
    num_auto = len([row for row in window if row["OutageType"] == "Auto"])
    bus_names = [row[bus] for row in window for bus in ["FirstBus", "SecondBus"]]
    num_unique_buses = len(set(bus_names))

    # fine-grained interval features
    fine_interval_features = []
    for dt in INTERVALS_MINUTES:
        dt = timedelta(minutes=dt)
        num_events_interval = len(
            [
                row
                for row in window
                if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < dt
            ]
        )
        fine_interval_features.append(num_events_interval)

    # graph based features
    node_degrees = np.array(
        [g.degree(bus_name_to_index[bus_name]) for bus_name in bus_names]
    )
    node_degrees_mean = node_degrees.mean().item()
    node_degrees_std = node_degrees.std().item()
    node_degrees_min = node_degrees.min().item()
    node_degrees_max = node_degrees.max().item()
    node_degrees_stats = [
        node_degrees_mean,
        node_degrees_std,
        node_degrees_min,
        node_degrees_max,
    ]

    # most recent planned outage features
    last_planned_voltage_group = VOLTAGE_GROUP[last_scheduled["Voltage"]]
    last_planned_voltage_one_hot = [
        int(voltage == last_planned_voltage_group) for voltage in VOLTAGES
    ]
    last_planned_node_degrees = np.array(
        [
            g.degree(bus_name_to_index[last_scheduled[bus]])
            for bus in ["FirstBus", "SecondBus"]
        ]
    )
    last_planned_node_degree_mean = last_planned_node_degrees.mean().item()

    # aggregate features
    features = [
        num_events,
        num_unique_ptids,
        *voltage_group_list,
        num_planned,
        num_auto,
        num_unique_buses,
        *fine_interval_features,
        *node_degrees_stats,
        *last_planned_voltage_one_hot,
        last_planned_node_degree_mean,
    ]

    ##########
    # labels #
    ##########
    # Auto/Planned labels
    is_auto = True if ref_row["OutageType"] == "Auto" else False

    # time to reference event
    time_to_event = ref_row["MinTimeStamp"] - max(row["MinTimeStamp"] for row in window)

    # location labels
    from_zone, to_zone = row_zones(ref_row)

    # aggregate labels
    labels = [is_auto, time_to_event.total_seconds(), from_zone, to_zone]

    return features, labels

In [10]:
from tqdm import tqdm

# reduce algorithm complexity by only looking at the most recent events in the window
MAX_EVENTS_PER_WINDOW = 200


def build_when_dataset(event_window_hours):
    dataset = []
    delta_time = timedelta(hours=event_window_hours)

    for i, auto_row in enumerate(
        tqdm(actual_data, desc=f"Window {event_window_hours:g}h")
    ):
        # exclude old data
        if (
            auto_row["MinTimeStamp"].year < MIN_YEAR
            or auto_row["OutDatetime"].year < MIN_YEAR
        ):
            continue

        # exclude planned event
        if auto_row["OutageType"] == "Planned":
            continue

        last_scheduled = actual_data[i - 1]
        if last_scheduled["OutageType"] == "Auto":
            continue

        # if scheduled outage ended before the automatic outage started, skip this sample
        if last_scheduled["MaxTimeStamp"] < auto_row["MinTimeStamp"]:
            continue

        window = [
            row
            for row in actual_data[max(i - MAX_EVENTS_PER_WINDOW, 0) : i]
            if last_scheduled["MinTimeStamp"] - row["MinTimeStamp"] < delta_time
        ]

        if window:
            dataset.append(create_when_sample(auto_row, window, last_scheduled))

    return dataset


# Build the first dataset for the inspection cells below. The remaining
# window sizes are built sequentially in the save cell to limit memory use.
when_dataset = build_when_dataset(EVENT_WINDOW_HOURS_VALUES[0])
len(when_dataset)

Window 0.25h: 100%|██████████| 70079/70079 [00:00<00:00, 374355.88it/s]


5396

In [11]:
features, labels = when_dataset[0]
features, labels

([1,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  2,
  0,
  0,
  0,
  1,
  3.0,
  0.0,
  3,
  3,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  3.0],
 [True, 3600.0, 12, 12])

In [12]:
from sklearn.model_selection import train_test_split, KFold


def assign_fold_ids(dataset, test_size=0.20, n_splits=5, seed=42):
    """
    Assign fold_id to each sample in a dataset.

    fold_id = -1  -> held-out test set
    fold_id = 0-4 -> 5-fold CV folds inside the remaining train/validation set

    Parameters
    ----------
    dataset : list-like
        Dataset where each item is like:
            features, label = dataset[i]

    test_size : float
        Proportion of data reserved as held-out test set.

    n_splits : int
        Number of CV folds.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    fold_ids : np.ndarray
        Array of shape (len(dataset),), containing fold IDs.
    """

    n_samples = len(dataset)
    indices = np.arange(n_samples)

    # Initialise all samples as unassigned
    fold_ids = np.empty(n_samples, dtype=int)

    # Step 1: create held-out test set
    trainval_idx, test_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=seed,
        shuffle=True,
    )

    # Assign held-out test samples
    fold_ids[test_idx] = -1

    # Step 2: assign 5-fold IDs inside train/validation set
    kfold = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    for fold_id, (_, val_idx_relative) in enumerate(kfold.split(trainval_idx)):
        val_idx_absolute = trainval_idx[val_idx_relative]
        fold_ids[val_idx_absolute] = fold_id

    return fold_ids

# save the dataset

In [13]:
from collections import Counter

feature_columns = [
    "num_events",
    "num_unique_ptids",
    *[f"num_{voltage}kv_lines" for voltage in VOLTAGES],
    "num_planned_outages",
    "num_auto_outages",
    "num_unique_buses",
    *[f"num_events_last_{minutes}_min" for minutes in INTERVALS_MINUTES],
    "node_degree_mean",
    "node_degree_std",
    "node_degree_min",
    "node_degree_max",
    *[f"last_planned_voltage_{voltage}kv" for voltage in VOLTAGES],
    "last_planned_node_degree_mean",
]
label_columns = [
    "label_is_auto",
    "label_time_to_event_seconds",
    "label_from_zone",
    "label_to_zone",
]
columns = ["fold_id", *feature_columns, *label_columns]

dataset_summaries = {}

for event_window_hours in EVENT_WINDOW_HOURS_VALUES:
    if event_window_hours == EVENT_WINDOW_HOURS_VALUES[0]:
        window_dataset = when_dataset
    else:
        window_dataset = build_when_dataset(event_window_hours)

    fold_ids = assign_fold_ids(window_dataset)
    dataset_csv_path = output_path / f"dataset_winsize{event_window_hours:g}h_when.csv"

    with dataset_csv_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(columns)
        for fold_id, (features, labels) in zip(fold_ids, window_dataset):
            if len(features) != len(feature_columns) or len(labels) != len(
                label_columns
            ):
                raise ValueError(
                    "Feature or label column count does not match the dataset sample shape."
                )
            writer.writerow([fold_id, *features, *labels])

    dataset_summaries[event_window_hours] = {
        "path": dataset_csv_path,
        "num_samples": len(window_dataset),
        "fold_counts": Counter(fold_ids),
    }

dataset_summaries

Window 12h: 100%|██████████| 70079/70079 [00:00<00:00, 304020.01it/s]


{0.25: {'path': PosixPath('output/dataset_winsize0.25h_when.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 0.5: {'path': PosixPath('output/dataset_winsize0.5h_when.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 1: {'path': PosixPath('output/dataset_winsize1h_when.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 2: {'path': PosixPath('output/dataset_winsize2h_when.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 86

# Provide "where" label

In [14]:
def create_where_sample(ref_row, window, last_scheduled):
    """Extract where-prediction context records and the reference-row zone label.

    Parameters
    ----------
    ref_row : pandas.Series or dict-like
        Reference outage row whose zones become the sample label.
    window : sequence of pandas.Series or dict-like
        Prior outage rows to convert into where-prediction context records.
    last_scheduled : pandas.Series or dict-like
        Most recent scheduled outage row. Kept for API compatibility and not used.

    Returns
    -------
    tuple[list[dict], dict]
        Window records and label. The label is the reference outage row's
        from_zone_index and to_zone_index.
    """
    ref_from_zone_index, ref_to_zone_index = row_zones(ref_row)
    label = {
        "from_zone_index": ref_from_zone_index,
        "to_zone_index": ref_to_zone_index,
    }
    window_records = [
        {
            "from_zone_index": from_zone_index,
            "to_zone_index": to_zone_index,
            "outage_type": row["OutageType"],
        }
        for row in window
        for from_zone_index, to_zone_index in [row_zones(row)]
    ]
    return window_records, label

In [15]:
import csv
import json
from collections import Counter


def build_where_dataset(event_window_hours):
    dataset = []
    delta_time = timedelta(hours=event_window_hours)

    for i, auto_row in enumerate(
        tqdm(actual_data, desc=f"Where window {event_window_hours:g}h")
    ):
        # auto_row: automatic outage event
        # keep this filtering aligned with build_when_dataset so fold IDs match
        if (
            auto_row["MinTimeStamp"].year < MIN_YEAR
            or auto_row["OutDatetime"].year < MIN_YEAR
        ):
            continue

        if auto_row["OutageType"] == "Planned":
            continue

        last_scheduled = actual_data[i - 1]
        if last_scheduled["OutageType"] == "Auto":
            continue

        # if scheduled outage ended before the automatic outage started, skip this sample
        if last_scheduled["MaxTimeStamp"] < auto_row["MinTimeStamp"]:
            continue

        window = [
            row
            for row in actual_data[max(i - MAX_EVENTS_PER_WINDOW, 0) : i]
            if last_scheduled["MinTimeStamp"] - row["MinTimeStamp"] < delta_time
        ]

        if window:
            dataset.append(create_where_sample(auto_row, window, last_scheduled))

    return dataset


def write_where_dataset_csv(dataset, fold_ids, dataset_csv_path):
    with dataset_csv_path.open("w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["fold_id", "window", "label"])
        for fold_id, (window_records, label) in zip(fold_ids, dataset):
            writer.writerow(
                [int(fold_id), json.dumps(window_records), json.dumps(label)]
            )


where_dataset_summaries = {}

for event_window_hours in EVENT_WINDOW_HOURS_VALUES:
    if event_window_hours == EVENT_WINDOW_HOURS_VALUES[0]:
        window_when_dataset = when_dataset
    else:
        window_when_dataset = build_when_dataset(event_window_hours)

    where_dataset = build_where_dataset(event_window_hours)
    where_fold_ids = assign_fold_ids(window_when_dataset)

    if len(where_dataset) != len(window_when_dataset):
        raise ValueError(
            "where_dataset and when_dataset must have the same sample count "
            f"for window {event_window_hours:g}h."
        )

    where_dataset_csv_path = (
        output_path / f"dataset_winsize{event_window_hours:g}h_where.csv"
    )
    write_where_dataset_csv(where_dataset, where_fold_ids, where_dataset_csv_path)

    where_dataset_summaries[event_window_hours] = {
        "path": where_dataset_csv_path,
        "num_samples": len(where_dataset),
        "fold_counts": Counter(where_fold_ids),
    }

where_dataset_summaries

Where window 12h: 100%|██████████| 70079/70079 [00:00<00:00, 806859.96it/s]


{0.25: {'path': PosixPath('output/dataset_winsize0.25h_where.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 0.5: {'path': PosixPath('output/dataset_winsize0.5h_where.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 1: {'path': PosixPath('output/dataset_winsize1h_where.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0): 864,
           np.int64(2): 863,
           np.int64(4): 863,
           np.int64(3): 863,
           np.int64(1): 863})},
 2: {'path': PosixPath('output/dataset_winsize2h_where.csv'),
  'num_samples': 5396,
  'fold_counts': Counter({np.int64(-1): 1080,
           np.int64(0)